<a href="https://colab.research.google.com/github/KTH-EXPECA/examples/blob/main/sdr/2sdr_2containers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set variables

In [ ]:
project = "stefan-project"
sdr_name1 = "sdr-01"
sdr_name2 = "sdr-02"
worker_name1 = "worker-03"
worker_name2 = "worker-03"


# Authentication and Dependencies

Login to Chameleon and download openrc.sh file from [here](https://testbed.expeca.proj.kth.se/project/api_access/openrc/). Upload it here next to this notebook and continue.

In the next cell, we setup the authentication method to be able to use Openstack clients.

In [ ]:
import os, re
from getpass import getpass

openrc_path = "/content/" + project + "-openrc.sh"

with open(openrc_path, 'r') as f:
    script_content = f.read()
    pattern = r'export\s+(\w+)\s*=\s*("[^"]+"|[^"\n]+)'
    matches = re.findall(pattern, script_content)

    for name, value in matches:
        os.environ[name] = value.strip('"')

password = getpass('enter your expeca password:')
os.environ['OS_PASSWORD'] = password

Install required packages and dependencies. Ignore the warnings.

In [ ]:
!pip uninstall -q -y moviepy
!pip install -q jedi
!pip install -q git+https://github.com/KTH-EXPECA/python-chi

Import packages

In [ ]:
import json, time
from loguru import logger
import chi.network, chi.container, chi.network
from chi.expeca import reserve, list_reservations, unreserve_byid, get_container_status, wait_until_container_removed, show_reservation_byname, restart_sdr, make_sdr_ni, make_sdr_mango, sdr_tools, get_available_publicips, get_segment_ids, get_radio_interfaces, get_worker_interfaces

# Reserve resources

In the next cells, we reserve 2 SDRs and 2 workers. Both workers can be the same, if desired.

In [ ]:
# Check the SDR's health and the status of its ports, both ports are supposed to be up, otherwise contact support
sdr_status = get_radio_interfaces(sdr_name1)
logger.info(f"{json.dumps(sdr_status, indent=4)}")
for port in sdr_status.keys():
  if sdr_status[port]['linkstate'] == 'Down':
    logger.warning(f"port {port} on {sdr_name1} is down.")
  if sdr_status[port]['linkstate'] == 'Up':
    logger.success(f"port {port} on {sdr_name1} is up.")

sdr_status = get_radio_interfaces(sdr_name2)
logger.info(f"{json.dumps(sdr_status, indent=4)}")
for port in sdr_status.keys():
  if sdr_status[port]['linkstate'] == 'Down':
    logger.warning(f"port {port} on {sdr_name2} is down.")
  if sdr_status[port]['linkstate'] == 'Up':
    logger.success(f"port {port} on {sdr_name2} is up.")


In [ ]:
# Reserve first SDR
segment_ids = get_segment_ids(sdr_name1)

# reserve RJ45 port
rj45_lease1 = show_reservation_byname(sdr_name1 + "-rj45-lease")
if not rj45_lease1:
    rj45_lease1 = reserve(
        { "type":"network", "name": sdr_name1+"-rj45", "net_name": sdr_name1+"-rj45", "segment_id": segment_ids['rj45'], "duration": { "days":7, "hours":0 } }
    )

# reserve SFP port
sfp_lease1 = show_reservation_byname(sdr_name1 + "-sfp-lease")
if not sfp_lease1:
    sfp_lease1 = reserve(
        { "type":"network", "name": sdr_name1+"-sfp", "net_name": sdr_name1+"-sfp", "segment_id": segment_ids['sfp'], "duration": { "days":7, "hours":0 } }
    )

# Reserve second SDR
segment_ids = get_segment_ids(sdr_name2)

# reserve RJ45 port
rj45_lease2 = show_reservation_byname(sdr_name2 + "-rj45-lease")
if not rj45_lease2:
    rj45_lease2 = reserve(
        { "type":"network", "name": sdr_name2+"-rj45", "net_name": sdr_name2+"-rj45", "segment_id": segment_ids['rj45'], "duration": { "days":7, "hours":0 } }
    )

# reserve SFP port
sfp_lease2 = show_reservation_byname(sdr_name2 + "-sfp-lease")
if not sfp_lease2:
    sfp_lease2 = reserve(
        { "type":"network", "name": sdr_name2+"-sfp", "net_name": sdr_name2+"-sfp", "segment_id": segment_ids['sfp'], "duration": { "days":7, "hours":0 } }
    )

# reserve first worker
worker_lease1 = show_reservation_byname(worker_name1+"-lease")
if not worker_lease1:
    worker_lease1 = reserve(
        { "type":"device", "name":worker_name1, "duration": { "days":7, "hours":0 } }
    )
worker_reservation_id1 = worker_lease1["reservations"][0]["id"]

# reserve second worker
worker_lease2 = show_reservation_byname(worker_name2+"-lease")
if not worker_lease2:
    worker_lease2 = reserve(
        { "type":"device", "name":worker_name2, "duration": { "days":7, "hours":0 } }
    )
worker_reservation_id2 = worker_lease2["reservations"][0]["id"]

leaseslist = list_reservations(brief=True)
print(json.dumps(leaseslist,indent=4))

# Run containers

Run a container for the first SDR SFP connection, with a public IP

In [ ]:
# check public IPs and select one
available_pub_ips = get_available_publicips()
if len(available_pub_ips) == 0:
  raise Exception("There is no available public IPs to reserve.")
pub_ip = available_pub_ips[0]
logger.info(f"Available public ips: {available_pub_ips}.")
logger.info(f"We choose {pub_ip} for this container.")

# check available interfaces of the worker
interfaces = list(get_worker_interfaces(worker_name1).values())[0]
available_ifs = []
for interface in interfaces.keys():
  if len(interfaces[interface]['connections']) == 0:
    available_ifs.append(interface)
logger.info(f"Available interfaces on {worker_name1}: {available_ifs}")

if len(available_ifs) < 2:
  logger.info(f"{json.dumps(interfaces, indent=4)}")
  raise Exception(f"Did not find enough interfaces on {worker_name1}")


# run the container
sdrsfpnet = chi.network.get_network(sdr_name1+"-sfp-net")
publicnet = chi.network.get_network("serverpublic")
container_name = sdr_name1 + "-sfp-container"
chi.container.create_container(
    name = container_name,
    image = "samiemostafavi/sshd-dind-oai",
    reservation_id = worker_reservation_id1,
    environment = {
        "DNS_IP":"8.8.8.8",
        "GATEWAY_IP":"130.237.11.97",
        "PASS":"expeca"
    },
    mounts = [],
    nets = [
        { "network" : publicnet['id'] },
        { "network" : sdrsfpnet['id'] },
    ],
    labels = {
        "networks.1.interface":available_ifs[0],
        "networks.1.ip":pub_ip+"/27",
        "capabilities.privileged":"true",
        "networks.2.interface":available_ifs[1],
        "networks.2.ip":"10.30.10.121/24",
    },
)
chi.container.wait_for_active(container_name)
logger.success(f"created {container_name} container, reachable at {pub_ip}.")

# install iperf3 + ip utilities in the container
command = "apt-get update && apt-get install -y iproute2 iputils-ping iperf3"
result = chi.container.execute(
    container_ref=container_name,
    command="curl -s -X POST -H \"Content-Type: application/json\" -d '{\"cmd\": \"" + command + "\"}' http://localhost:50505/",
)
logger.info(f"{result}")


Run a container for the second SDR SFP connection, with a public IP

In [ ]:
# check public IPs and select one
available_pub_ips = get_available_publicips()
if len(available_pub_ips) == 0:
  raise Exception("There is no available public IPs to reserve.")
pub_ip = available_pub_ips[0]
logger.info(f"Available public ips: {available_pub_ips}.")
logger.info(f"We choose {pub_ip} for this container.")

# check available interfaces of the worker
interfaces = list(get_worker_interfaces(worker_name2).values())[0]
available_ifs = []
for interface in interfaces.keys():
  if len(interfaces[interface]['connections']) == 0:
    available_ifs.append(interface)
logger.info(f"Available interfaces on {worker_name2}: {available_ifs}")

if len(available_ifs) < 2:
  logger.info(f"{json.dumps(interfaces, indent=4)}")
  raise Exception(f"Did not find enough interfaces on {worker_name2}")


# run the container
sdrsfpnet = chi.network.get_network(sdr_name2+"-sfp-net")
publicnet = chi.network.get_network("serverpublic")
container_name = sdr_name2 + "-sfp-container"
chi.container.create_container(
    name = container_name,
    image = "samiemostafavi/sshd-dind-oai",
    reservation_id = worker_reservation_id2,
    environment = {
        "DNS_IP":"8.8.8.8",
        "GATEWAY_IP":"130.237.11.97",
        "PASS":"expeca"
    },
    mounts = [],
    nets = [
        { "network" : publicnet['id'] },
        { "network" : sdrsfpnet['id'] },
    ],
    labels = {
        "networks.1.interface":available_ifs[0],
        "networks.1.ip":pub_ip+"/27",
        "capabilities.privileged":"true",
        "networks.2.interface":available_ifs[1],
        "networks.2.ip":"10.30.10.122/24",
    },
)
chi.container.wait_for_active(container_name)
logger.success(f"created {container_name} container, reachable at {pub_ip}.")

# install iperf3 + ip utilities in the container
command = "apt-get update && apt-get install -y iproute2 iputils-ping iperf3"
result = chi.container.execute(
    container_ref=container_name,
    command="curl -s -X POST -H \"Content-Type: application/json\" -d '{\"cmd\": \"" + command + "\"}' http://localhost:50505/",
)
logger.info(f"{result}")
